# What is a Dubai home worth? — an honest price model on 1.38M official DLD sales

**Question.** Given only what is known on the day a sale is registered with the Dubai Land Department
(area, property type, off-plan vs ready, size, rooms, parking, and the *past* prices of comparable sales),
how well can we predict the registered **price per square metre** — and therefore the price?

**Data.** Dubai Land Department open data via *Dubai Real Estate Data* (dubairealestatedata.com), CC BY 4.0.
Every row is a registered sale, not an asking price.

**Rules this notebook follows**
* Every history feature for a sale on day *D* uses only sales registered **before** *D* (not the same day).
* Time-based split: train ≤ 2023, validation = 2024, test = 2025-01-01 → 2026-11-02, touched once at the end.
* Every number reported anywhere in the project is written to `results/metrics.json` by this notebook.

**Kaggle version.** The project's `src/` modules are written below with `%%writefile` so this notebook runs on its own. Attach the dataset *tomasgrygar/dubai-real-estate-sales-transactions* via **Add Input** — the path under `/kaggle/input` is found automatically.

Source repo: https://github.com/HussainAhmed75/dubai-property-price-model

In [ ]:
import os
os.makedirs('/kaggle/working/src', exist_ok=True)
os.chdir('/kaggle/working')

In [ ]:
%%writefile /kaggle/working/src/features.py
"""Data loading, cleaning and leakage-safe feature engineering for the
Dubai DLD sales-transactions dataset.

Every "history" feature for a transaction registered on day D is computed
ONLY from transactions registered on days strictly before D (closed='left'
rolling windows on a daily aggregate). Same-day transactions never see each
other, so nothing from the future (or the present day) leaks into a row.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

SQM_TO_SQFT = 10.7639

# Residential sub-types we keep (others inside usage=Residential are offices,
# shops, hotel rooms ... which price on a different logic).
KEEP_SUBTYPES = {"Flat", "Villa", "Stacked Townhouses", "Unit"}

MODEL_START = "2013-01-01"   # first date used for training rows
HISTORY_START = "2010-01-01"  # earlier rows only feed the rolling history

CATEGORICAL = ["reg_type", "prop_type", "procedure_group", "area_id_cat"]

NUMERIC = [
    "log_area_sqm", "rooms", "has_parking", "month",
    "area_mean_365", "area_n_365", "area_mean_90",
    "seg_mean_365", "seg_n_365",
    "rooms_mean_365",
    "proj_mean_365", "proj_n_365", "proj_mean_all", "proj_n_all",
    "bldg_mean_all", "bldg_n_all",
    "mkt_mean_30", "mkt_reg_mean_30",
    "proj_age_days",
]
FEATURES = NUMERIC + CATEGORICAL
TARGET = "log_psm"   # log(price per square metre, AED)


def find_data_file(candidates=None) -> str:
    """Locate the parquet file locally or under /kaggle/input."""
    import glob
    import os
    candidates = candidates or [
        "data/dld_sales_transactions.parquet",
        "../data/dld_sales_transactions.parquet",
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    hits = glob.glob("/kaggle/input/**/dld_sales_transactions.parquet", recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError("dld_sales_transactions.parquet not found")


def load_raw(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:
    """Apply documented cleaning steps; return cleaned frame + a step log."""
    log: list[dict] = []

    def step(name, reason, new):
        log.append({"step": name, "reason": reason,
                    "rows_before": int(len(cur)), "rows_after": int(len(new)),
                    "rows_removed": int(len(cur) - len(new))})
        return new

    cur = df.copy()
    log.append({"step": "raw", "reason": "file as downloaded",
                "rows_before": int(len(cur)), "rows_after": int(len(cur)), "rows_removed": 0})
    cur["date"] = pd.to_datetime(cur["instance_date"], errors="coerce")

    cur = step("quality_flags == 0",
               "publisher's own checks: price floor, area range, P1-P99 psf band, bad dates",
               cur[cur["quality_flags"] == 0])
    cur = step("residential Unit/Villa only",
               "Land trades on plot area and whole Buildings price wholesale; commercial units price differently",
               cur[cur["property_type_label"].isin(["Unit", "Villa"]) & (cur["usage_label"] == "Residential")])
    cur = step("drop non-home sub-types",
               "offices/shops/hotel rooms/hotel apartments registered under Residential",
               cur[cur["property_sub_type"].isna() | cur["property_sub_type"].isin(KEEP_SUBTYPES)])
    dup_cols = [c for c in df.columns if c != "transaction_id"]
    cur = step("drop exact duplicates (all columns except transaction_id)",
               "cannot tell identical bulk sales from re-registrations; keep one so no single deal dominates",
               cur.drop_duplicates(dup_cols))
    cur = step(f"date >= {HISTORY_START}",
               "pre-2010 years are thin (<10k rows/yr before 2007) and only add noise to rolling history",
               cur[cur["date"] >= HISTORY_START])

    cur = cur.sort_values(["date", "transaction_id"]).reset_index(drop=True)
    cur["psm"] = cur["price_aed"] / cur["area_sqm"]
    cur[TARGET] = np.log(cur["psm"])
    return cur, log


def _procedure_group(s: pd.Series) -> pd.Series:
    s = s.fillna("Unknown")
    out = np.where(s.str.contains("Lease to Own"), "LeaseToOwn",
          np.where(s.str.contains("Development"), "Development",
          np.where(s.str.contains("Delayed"), "Delayed",
          np.where(s.str.contains("Payment Plan"), "PaymentPlan",
          np.where(s.str.startswith("Sell"), "Sell", "Other")))))
    return pd.Series(out, index=s.index)


def _past_rolling(df: pd.DataFrame, keys: list[str], window: str | None,
                  value: str = TARGET) -> pd.DataFrame:
    """Mean and count of `value` for each key over the trailing `window`
    of days STRICTLY BEFORE each row's date (window=None -> all past days).

    Implementation: aggregate to (key, day) sums/counts, then a left-closed
    time rolling window per key, then join back on (key, day).
    """
    g = (df.groupby(keys + ["date"], observed=True, sort=True)[value]
           .agg(["sum", "count"]).reset_index())
    if window is None:
        grp = g.groupby(keys, observed=True, sort=False)
        g["s"] = grp["sum"].cumsum() - g["sum"]      # exclude current day
        g["n"] = grp["count"].cumsum() - g["count"]
    else:
        parts = []
        for _, sub in g.groupby(keys, observed=True, sort=False):
            r = sub.set_index("date")[["sum", "count"]].rolling(window, closed="left").sum()
            sub = sub.copy()
            sub["s"] = r["sum"].to_numpy()
            sub["n"] = r["count"].to_numpy()
            parts.append(sub)
        g = pd.concat(parts)
    g["n"] = g["n"].fillna(0)
    g["mean"] = np.where(g["n"] > 0, g["s"] / g["n"].replace(0, np.nan), np.nan)
    out = df[keys + ["date"]].merge(g[keys + ["date", "mean", "n"]],
                                    on=keys + ["date"], how="left")
    return out[["mean", "n"]].set_axis(df.index)


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the cleaned frame with all model features added."""
    d = df.copy()
    d["reg_type"] = d["reg_type_label"].astype("category")
    d["prop_type"] = d["property_type_label"].astype("category")
    d["procedure_group"] = _procedure_group(d["procedure_name_en"]).astype("category")
    d["area_id_cat"] = d["area_id"].astype("Int64").astype(str).astype("category")
    d["log_area_sqm"] = np.log(d["area_sqm"])
    d["rooms"] = d["rooms_bucket"].astype(float)
    d["rooms_key"] = d["rooms_bucket"].fillna(-1).astype(int)
    d["has_parking"] = d["has_parking"].map({True: 1.0, False: 0.0}).astype(float)
    d["month"] = d["date"].dt.month.astype(float)
    d["proj_key"] = d["project_name"].fillna("__none__").str.strip().str.lower()
    d["bldg_key"] = d["building_name"].fillna("__none__").str.strip().str.lower()
    d["all_key"] = 0

    def put(prefix, keys, window):
        r = _past_rolling(d, keys, window)
        d[f"{prefix}_mean"] = r["mean"]
        d[f"{prefix}_n"] = r["n"]

    put("area_365", ["area_id"], "365D")
    put("area_90", ["area_id"], "90D")
    put("seg_365", ["area_id", "property_type_label", "reg_type_label"], "365D")
    put("rooms_365", ["area_id", "rooms_key"], "365D")
    put("proj_365", ["proj_key"], "365D")
    put("proj_all", ["proj_key"], None)
    put("bldg_all", ["bldg_key"], None)
    put("mkt_30", ["all_key"], "30D")
    put("mkt_reg_30", ["reg_type_label"], "30D")

    d = d.rename(columns={
        "area_365_mean": "area_mean_365", "area_365_n": "area_n_365",
        "area_90_mean": "area_mean_90",
        "seg_365_mean": "seg_mean_365", "seg_365_n": "seg_n_365",
        "rooms_365_mean": "rooms_mean_365",
        "proj_365_mean": "proj_mean_365", "proj_365_n": "proj_n_365",
        "proj_all_mean": "proj_mean_all", "proj_all_n": "proj_n_all",
        "bldg_all_mean": "bldg_mean_all", "bldg_all_n": "bldg_n_all",
        "mkt_30_mean": "mkt_mean_30", "mkt_reg_30_mean": "mkt_reg_mean_30",
    })
    # "no project / no building" is not a real group -> blank its history
    none_p = d["proj_key"] == "__none__"
    d.loc[none_p, ["proj_mean_365", "proj_mean_all"]] = np.nan
    d.loc[none_p, ["proj_n_365", "proj_n_all"]] = 0
    none_b = d["bldg_key"] == "__none__"
    d.loc[none_b, "bldg_mean_all"] = np.nan
    d.loc[none_b, "bldg_n_all"] = 0

    # project age: days since the project's first sale strictly before today
    first = (d[~none_p].groupby("proj_key")["date"].min())
    d["proj_age_days"] = (d["date"] - d["proj_key"].map(first)).dt.days.astype(float)
    d.loc[none_p, "proj_age_days"] = np.nan
    return d


In [ ]:
%%writefile /kaggle/working/src/modeling.py
"""Time-based splits, baselines, models, tuning and evaluation."""
from __future__ import annotations

import time

import numpy as np
import pandas as pd

from features import CATEGORICAL, FEATURES, NUMERIC, TARGET

SEED = 42
TRAIN_END = "2023-12-31"
VAL_START, VAL_END = "2024-01-01", "2024-12-31"
TEST_START = "2025-01-01"
N_THREADS = 2


# ---------------------------------------------------------------- splits
def time_split(d: pd.DataFrame, model_start: str = "2013-01-01"):
    tr = d[(d["date"] >= model_start) & (d["date"] <= TRAIN_END)]
    va = d[(d["date"] >= VAL_START) & (d["date"] <= VAL_END)]
    te = d[d["date"] >= TEST_START]
    assert tr["date"].max() < va["date"].min() <= va["date"].max() < te["date"].min()
    return tr, va, te


# ---------------------------------------------------------------- metrics
def price_metrics(y_log_true, y_log_pred) -> dict:
    """Errors expressed on the transaction price. Because area is known,
    % error on price == % error on price-per-sqm."""
    yt = np.exp(np.asarray(y_log_true, dtype=float))
    yp = np.exp(np.asarray(y_log_pred, dtype=float))
    ape = np.abs(yp - yt) / yt
    lt, lp = np.asarray(y_log_true, float), np.asarray(y_log_pred, float)
    ss_res = np.sum((lt - lp) ** 2)
    ss_tot = np.sum((lt - lt.mean()) ** 2)
    return {
        "mdape_pct": float(np.median(ape) * 100),
        "mape_pct": float(np.mean(ape) * 100),
        "within_10pct": float(np.mean(ape <= 0.10) * 100),
        "within_20pct": float(np.mean(ape <= 0.20) * 100),
        "r2_log_psm": float(1 - ss_res / ss_tot),
        "mae_log": float(np.mean(np.abs(lt - lp))),
        "n": int(len(lt)),
    }


# ---------------------------------------------------------------- baselines
def baseline_market(d: pd.DataFrame) -> np.ndarray:
    """Naive: last-30-day market average (same registration type)."""
    return d["mkt_reg_mean_30"].fillna(d["mkt_mean_30"]).to_numpy()


def baseline_comps(d: pd.DataFrame) -> np.ndarray:
    """Agent-style 'comps' rule, ONE strong feature with fallbacks:
    same area + property type + off-plan/ready over the past 365 days,
    else whole area, else market."""
    return (d["seg_mean_365"].fillna(d["area_mean_365"])
            .fillna(d["mkt_reg_mean_30"]).fillna(d["mkt_mean_30"]).to_numpy())


def baseline_building(d: pd.DataFrame) -> np.ndarray:
    """Stronger hand-made rule: past sales in the same building, else same
    project (365d), else the comps rule."""
    return (d["bldg_mean_all"].where(d["bldg_n_all"] >= 3)
            .fillna(d["proj_mean_365"].where(d["proj_n_365"] >= 3))
            .fillna(pd.Series(baseline_comps(d), index=d.index)).to_numpy())


# ---------------------------------------------------------------- matrices
def X_tree(d: pd.DataFrame) -> pd.DataFrame:
    return d[FEATURES]


def X_codes(d: pd.DataFrame, cats_ref: pd.DataFrame) -> pd.DataFrame:
    """Integer-coded categoricals (for RandomForest)."""
    X = d[FEATURES].copy()
    for c in CATEGORICAL:
        X[c] = pd.Categorical(X[c], categories=cats_ref[c].cat.categories).codes.astype(float)
        X.loc[X[c] < 0, c] = np.nan
    return X


# ---------------------------------------------------------------- models
def make_linear():
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import Ridge
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                    ("sc", StandardScaler())])
    cat = OneHotEncoder(handle_unknown="ignore", min_frequency=20)
    pre = ColumnTransformer([("num", num, NUMERIC), ("cat", cat, CATEGORICAL)])
    return Pipeline([("pre", pre), ("ridge", Ridge(alpha=1.0))])


def lgbm_params(**over):
    p = dict(objective="regression", learning_rate=0.05, n_estimators=3000,
             num_leaves=127, min_child_samples=40, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED,
             n_jobs=N_THREADS, deterministic=True, force_row_wise=True, verbose=-1)
    p.update(over)
    return p


def fit_lgbm(Xtr, ytr, Xva, yva, **over):
    import lightgbm as lgb
    m = lgb.LGBMRegressor(**lgbm_params(**over))
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m


def fit_xgb(Xtr, ytr, Xva, yva):
    import xgboost as xgb
    m = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.05, max_depth=8,
                         subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                         tree_method="hist", enable_categorical=True,
                         max_cat_to_onehot=1, random_state=SEED, n_jobs=N_THREADS,
                         early_stopping_rounds=100)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m


def fit_rf(Xtr, ytr, n_sub=300_000):
    from sklearn.ensemble import RandomForestRegressor
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(Xtr), size=min(n_sub, len(Xtr)), replace=False)
    m = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, max_features=0.5,
                              n_jobs=N_THREADS, random_state=SEED)
    m.fit(Xtr.iloc[idx], np.asarray(ytr)[idx])
    return m


def timed(fn, *a, **k):
    t = time.time()
    out = fn(*a, **k)
    return out, round(time.time() - t, 1)


# ---------------------------------------------------------------- tuning
def tune_lgbm(Xtr, ytr, Xva, yva, n_trials=25):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        over = dict(
            learning_rate=0.08,
            num_leaves=trial.suggest_int("num_leaves", 31, 511, log=True),
            min_child_samples=trial.suggest_int("min_child_samples", 10, 300, log=True),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
            min_split_gain=trial.suggest_float("min_split_gain", 0.0, 0.05),
        )
        m = fit_lgbm(Xtr, ytr, Xva, yva, **over)
        pred = m.predict(Xva)
        trial.set_user_attr("best_iter", int(m.best_iteration_))
        return price_metrics(yva, pred)["mdape_pct"]

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=n_trials)
    return study


# ---------------------------------------------------------------- intervals
def split_conformal(resid_val: np.ndarray, alpha: float = 0.2) -> float:
    """Half-width (in log space) of a (1-alpha) interval from validation residuals."""
    r = np.abs(np.asarray(resid_val))
    n = len(r)
    q = np.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(r, min(q, 1.0)))


In [ ]:
%%writefile /kaggle/working/src/style.py
"""One consistent visual theme for every static chart (matplotlib) and the
Plotly dashboard."""
from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt

# palette: deep navy ink, one sand/gold accent (Dubai), muted greys
INK = "#1B2A41"
ACCENT = "#C8963E"      # highlight / the model
ACCENT_2 = "#2E7D7A"    # second series (e.g. Ready vs Off-Plan)
MUTED = "#A7B0BE"       # baselines / context
GRID = "#E6E9EF"
BG = "#FFFFFF"
GOOD = "#2E7D32"
BAD = "#B23A48"

SERIES = {"Off-Plan": ACCENT, "Ready": ACCENT_2}


def apply():
    mpl.rcParams.update({
        "figure.figsize": (10, 5.6),
        "figure.dpi": 110,
        "savefig.dpi": 160,
        "savefig.bbox": "tight",
        "figure.facecolor": BG,
        "axes.facecolor": BG,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 26,
        "axes.labelsize": 11,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.color": INK,
        "ytick.color": INK,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.frameon": False,
        "font.family": "DejaVu Sans",
        "text.color": INK,
    })


def title(ax, headline: str, sub: str | None = None):
    """Headline = the takeaway; sub = what is plotted."""
    ax.set_title(headline)
    if sub:
        ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=10.5,
                color="#5B6676", va="bottom", ha="left")


def source(fig, text="Source: Dubai Land Department open data via dubairealestatedata.com (CC BY 4.0)"):
    fig.text(0.01, -0.02, text, fontsize=8, color="#7A8494", ha="left")


def save(fig, path):
    fig.savefig(path)
    plt.close(fig)


PLOTLY_LAYOUT = dict(
    font=dict(family="DejaVu Sans, Arial, sans-serif", color=INK, size=13),
    paper_bgcolor=BG, plot_bgcolor=BG,
    xaxis=dict(gridcolor=GRID, zeroline=False),
    yaxis=dict(gridcolor=GRID, zeroline=False),
    margin=dict(l=60, r=30, t=80, b=50),
    colorway=[ACCENT, ACCENT_2, INK, MUTED, BAD, GOOD],
)


In [ ]:
%%writefile /kaggle/working/src/make_dashboard.py
"""Builds dashboard/dubai_price_dashboard.html — a single self-contained,
offline Plotly dashboard. Every number comes from results/metrics.json or
results/test_predictions.csv.gz."""
from __future__ import annotations

import html
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import style  # noqa: E402


def _layout(fig, title, sub=None, height=460):
    t = f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:#5B6676'>{sub}</span>" if sub else "")
    fig.update_layout(**style.PLOTLY_LAYOUT, title=dict(text=t, x=0.01, xanchor="left"), height=height)
    return fig


def build(root: str) -> str:
    M = json.load(open(os.path.join(root, "results", "metrics.json"), encoding="utf-8"))
    te = pd.read_csv(os.path.join(root, "results", "test_predictions.csv.gz"), parse_dates=["date"])
    te["ym"] = te["date"].dt.to_period("M").astype(str)
    te["actual_psm"] = te["price_aed"] / te["area_sqm"]
    te["pred_psm"] = te["pred_price"] / te["area_sqm"]
    figs = []

    # 1. model comparison
    names = [n for n in ["Market baseline", "Comps rule (1 feature)", "Ridge (linear)", "Building rule",
                         "PerpetualBooster", "Random Forest", "XGBoost", "LightGBM"] if n in M["validation"]]
    tn = [n if n != "LightGBM" else "LightGBM (tuned, final)" for n in names]
    fig = go.Figure([
        go.Bar(name="Validation 2024", x=names, y=[M["validation"][n]["mdape_pct"] for n in names], marker_color=style.MUTED,
               text=[f"{M['validation'][n]['mdape_pct']:.1f}%" for n in names], textposition="outside"),
        go.Bar(name="Test 2025-26", x=names, y=[M["test"][n]["mdape_pct"] for n in tn], marker_color=style.ACCENT,
               text=[f"{M['test'][n]['mdape_pct']:.1f}%" for n in tn], textposition="outside"),
    ])
    wf = M["walk_forward"]["mdape_pct"]
    fig.add_trace(go.Bar(name="Test 2025-26, retrained monthly", x=["LightGBM retrained monthly"], y=[wf],
                         marker_color=style.ACCENT_2, text=[f"{wf:.1f}%"], textposition="outside"))
    fig.update_traces(textfont_size=11)
    fig.update_layout(barmode="group", bargap=0.25, yaxis_title="Median abs. % error (lower is better)", uniformtext=dict(minsize=10, mode="show"))
    figs.append(_layout(fig, "Model vs baselines",
                        "LightGBM bar on test = tuned final model; others on test are for reference only"))

    # 2. error by month
    bm = pd.DataFrame(M["test_by_month"]).T
    fig = go.Figure([
        go.Scatter(x=bm.index, y=bm["mdape_building_rule"], name="Building rule", mode="lines+markers", line=dict(color=style.MUTED, width=2)),
        go.Scatter(x=bm.index, y=bm["mdape_model"], name="Model frozen at end of 2024", mode="lines+markers", line=dict(color=style.BAD, width=2.5)),
        go.Scatter(x=bm.index, y=bm["mdape_walk_forward"], name="Model retrained monthly", mode="lines+markers", line=dict(color=style.ACCENT, width=3)),
    ])
    fig.update_layout(yaxis_title="Median abs. % error", yaxis_rangemode="tozero")
    figs.append(_layout(fig, "Error on unseen months (2025 → Nov 2026)", "A frozen model ages; retraining each month keeps it ahead of the rule"))

    # 3. results explorer by area
    top_areas = te["area_name_en"].value_counts().head(20).index.tolist()
    fig = go.Figure()
    buttons = []
    for i, ar in enumerate(top_areas):
        g = te[te.area_name_en == ar].groupby("ym").agg(actual=("actual_psm", "median"), pred=("pred_psm", "median"), n=("actual_psm", "size"))
        vis = i == 0
        fig.add_trace(go.Scatter(x=g.index, y=g["actual"], name="Actual median AED/m²", visible=vis,
                                 mode="lines+markers", line=dict(color=style.ACCENT_2, width=3),
                                 customdata=g["n"], hovertemplate="%{x}<br>actual %{y:,.0f} AED/m²<br>%{customdata} sales<extra></extra>"))
        fig.add_trace(go.Scatter(x=g.index, y=g["pred"], name="Predicted median AED/m²", visible=vis,
                                 mode="lines+markers", line=dict(color=style.ACCENT, width=3, dash="dot"),
                                 hovertemplate="%{x}<br>predicted %{y:,.0f} AED/m²<extra></extra>"))
        mask = [False] * (2 * len(top_areas)); mask[2 * i] = mask[2 * i + 1] = True
        sub = te[te.area_name_en == ar]
        md = float(np.median(np.abs(sub["signed_err_pct"])))
        buttons.append(dict(label=ar, method="update",
                            args=[{"visible": mask}, {"title.text": f"<b>Results explorer — {html.escape(ar)}</b><br><span style='font-size:12px;color:#5B6676'>{len(sub):,} test sales · median abs. error {md:.1f}%</span>"}]))
    fig.update_layout(updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.18, yanchor="top")],
                      yaxis_title="AED per m²")
    fig.add_annotation(text=f"{M['drift']['last_month']} has only {M['drift']['last_month_is_partial_days']} days of data",
                       xref="paper", yref="paper", x=0, y=-0.2, showarrow=False, font=dict(size=11, color="#7A8494"))
    sub0 = te[te.area_name_en == top_areas[0]]
    figs.append(_layout(fig, f"Results explorer — {top_areas[0]}",
                        f"{len(sub0):,} test sales · median abs. error {np.median(np.abs(sub0['signed_err_pct'])):.1f}%", height=500))
    fig.update_layout(margin=dict(b=95))

    # 4. signed error distribution by off-plan / ready
    fig = go.Figure()
    for k, col in [("Ready", style.ACCENT_2), ("Off-Plan", style.ACCENT)]:
        e = te.loc[te.reg_type_label == k, "signed_err_pct"].clip(-60, 60)
        fig.add_trace(go.Histogram(x=e, name=k, nbinsx=120, marker_color=col, opacity=0.65, histnorm="percent"))
    fig.update_layout(barmode="overlay", xaxis_title="Predicted vs actual price (%; clipped at ±60)", yaxis_title="% of sales")
    figs.append(_layout(fig, "How far off are the predictions?", "Negative = model below the registered price"))

    # 5. area growth
    ag = pd.DataFrame(M["area_growth_2021_2025_top10"]).iloc[::-1]
    fig = go.Figure(go.Bar(x=ag["growth_pct"], y=ag["area"], orientation="h", marker_color=style.ACCENT,
                           text=[f"+{v:.0f}%" for v in ag["growth_pct"]], textposition="outside", cliponaxis=False,
                           customdata=np.c_[ag["psm_2021"], ag["psm_2025"]],
                           hovertemplate="%{y}<br>2021: %{customdata[0]:,} AED/m²<br>2025: %{customdata[1]:,} AED/m²<extra></extra>"))
    fig.update_layout(xaxis_title="Median AED/m² change 2021 → 2025", xaxis_range=[0, ag["growth_pct"].max() * 1.15])
    figs.append(_layout(fig, "Fastest-rising areas (≥300 sales in 2021 and 2025)", height=480))

    h = M["headline"]; d = M["dataset"]; iv = M["interval"]
    kpis = [
        (f"{d['rows_raw']:,}", "registered sales in the raw file"),
        (f"{h['test_mdape_pct']:.1f}%", "median error, model frozen at end of 2024"),
        (f"{h['test_within_10pct']:.1f}%", "of test sales within ±10%"),
        (f"{h['building_rule_test_mdape_pct']:.1f}%", "error of the building rule"),
        (f"{M['walk_forward']['mdape_pct']:.1f}%", "median error if retrained monthly"),
        (f"{iv['test_coverage_pct']:.1f}%", f"real coverage of the 80% interval"),
    ]
    case_rows = "".join(
        f"<tr><td>{c['case']}</td><td>{c['date']}</td><td>{html.escape(str(c['area']))}</td><td>{c['reg_type']} {c['type']}</td>"
        f"<td>{c['size_sqm']}</td><td>{c['actual_aed']:,}</td><td>{c['predicted_aed']:,}</td><td>{c['error_pct']:+.1f}%</td></tr>"
        for c in M["case_studies"])
    parts = []
    for i, f in enumerate(figs):
        parts.append(f.to_html(full_html=False, include_plotlyjs="inline" if i == 0 else False,
                               config={"displaylogo": False, "responsive": True}))
    page = f"""<!DOCTYPE html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>Dubai home prices — model dashboard</title>
<style>body{{font-family:'DejaVu Sans',Arial,sans-serif;color:{style.INK};margin:0;background:#F7F8FA}}
.wrap{{max-width:1150px;margin:0 auto;padding:28px}} h1{{margin:0 0 4px}} .sub{{color:#5B6676;margin-bottom:22px}}
.kpis{{display:grid;grid-template-columns:repeat(6,1fr);gap:12px;margin-bottom:18px}}
.kpi{{background:#fff;border-radius:10px;padding:14px;border:1px solid {style.GRID}}} .kpi b{{font-size:26px;color:{style.ACCENT}}}
.kpi span{{display:block;font-size:13px;color:#5B6676}} .card{{background:#fff;border-radius:10px;padding:8px;border:1px solid {style.GRID};margin-bottom:16px}}
table{{border-collapse:collapse;width:100%;font-size:13px}} td,th{{border-bottom:1px solid {style.GRID};padding:6px;text-align:left}}
.foot{{font-size:12px;color:#7A8494}} @media(max-width:800px){{.kpis{{grid-template-columns:repeat(2,1fr)}}}}</style></head>
<body><div class="wrap"><h1>What is a Dubai home worth?</h1>
<div class="sub">Price model on official Dubai Land Department sales · trained ≤2024 · tested on {M['splits']['test']['from']} → {M['splits']['test']['to']}</div>
<div class="kpis">{''.join(f'<div class="kpi"><b>{a}</b><span>{b}</span></div>' for a, b in kpis)}</div>
{''.join(f'<div class="card">{p}</div>' for p in parts)}
<div class="card" style="padding:16px"><h3>Case studies (real test-period sales)</h3><table>
<tr><th>Case</th><th>Date</th><th>Area</th><th>Type</th><th>m²</th><th>Actual AED</th><th>Predicted AED</th><th>Error</th></tr>{case_rows}</table></div>
<p class="foot">Data: Dubai Land Department open data via dubairealestatedata.com (CC BY 4.0). Code: MIT. Built by Hussain Ahmed.</p>
</div></body></html>"""
    out = os.path.join(root, "dashboard", "dubai_price_dashboard.html")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    open(out, "w", encoding="utf-8").write(page)
    return out


if __name__ == "__main__":
    build(sys.argv[1] if len(sys.argv) > 1 else os.path.dirname(os.path.dirname(os.path.abspath(__file__))))


In [ ]:
import os, sys, json, glob, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

ON_KAGGLE = os.path.exists("/kaggle/input")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
for d in ["results", "figures", "models"]:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

from features import (FEATURES, NUMERIC, CATEGORICAL, TARGET, SQM_TO_SQFT,
                      find_data_file, load_raw, clean, build_features)
import modeling as mdl
import style
style.apply()
import matplotlib.pyplot as plt

DATA = find_data_file([os.path.join(ROOT, "data", "dld_sales_transactions.parquet")])
R = lambda *p: os.path.join(ROOT, *p)
METRICS = {}
print("data file:", DATA, "| on kaggle:", ON_KAGGLE)

## 1. Load and inspect

In [ ]:
raw = load_raw(DATA)
print(raw.shape)
display(raw.head(3).T)
info = pd.DataFrame({"dtype": raw.dtypes.astype(str),
                     "missing_pct": (raw.isna().mean() * 100).round(2),
                     "n_unique": raw.nunique()})
display(info)
dups_all_but_id = int(raw.duplicated([c for c in raw.columns if c != "transaction_id"]).sum())
print("transaction_id unique:", raw["transaction_id"].is_unique)
print("rows identical except transaction_id:", dups_all_but_id)
print(raw["quality_flags"].value_counts().sort_index())

In [ ]:
# odd values worth knowing about
d0 = pd.to_datetime(raw["instance_date"], errors="coerce")
odd = {
    "dates_before_2000": int((d0 < "2000-01-01").sum()),
    "max_date": str(d0.max().date()),
    "price_aed_max": int(raw["price_aed"].max()),
    "area_sqm_max_residential_unit": float(raw.loc[(raw.property_type_label == "Unit") & (raw.usage_label == "Residential"), "area_sqm"].max()),
    "rooms_missing_pct": float(raw["rooms_bucket"].isna().mean() * 100),
}
odd

## 2. Cleaning — every decision, its reason, and the rows it removed

In [ ]:
df, clean_log = clean(raw)
log_df = pd.DataFrame(clean_log)
display(log_df)
METRICS["dataset"] = {
    "rows_raw": int(len(raw)),
    "rows_clean": int(len(df)),
    "date_min_clean": str(df["date"].min().date()),
    "date_max_clean": str(df["date"].max().date()),
    "n_areas": int(df["area_id"].nunique()),
    "duplicates_all_but_id_raw": dups_all_but_id,
    "license": "CC BY 4.0",
    "source": "Dubai Land Department open data via dubairealestatedata.com",
}
METRICS["cleaning_log"] = clean_log

## 3. EDA — facts worth telling

In [ ]:
df["year"] = df["date"].dt.year
yr = df.groupby("year")
by_year = pd.DataFrame({
    "transactions": yr.size(),
    "offplan_share_pct": yr.apply(lambda g: (g["reg_type_label"] == "Off-Plan").mean() * 100),
    "median_psm": yr["psm"].median(),
    "median_size_sqm": yr["area_sqm"].median(),
})
display(by_year.round(1))

y25 = df[df.year == 2025]
reg25 = y25.groupby("reg_type_label")["psm"].median()
a = (df[df.year.isin([2021, 2025])].groupby(["area_name_en", "year"])["psm"]
       .agg(["median", "size"]).unstack())
a = a[(a["size"] >= 300).all(axis=1)]
a = pd.DataFrame({"psm_2021": a[("median", 2021)], "psm_2025": a[("median", 2025)],
                  "n_2021": a[("size", 2021)], "n_2025": a[("size", 2025)]})
a["growth_pct"] = (a["psm_2025"] / a["psm_2021"] - 1) * 100
a = a.sort_values("growth_pct", ascending=False)
top_area_25 = y25["area_name_en"].value_counts()

facts = {
    "tx_2020": int(by_year.loc[2020, "transactions"]),
    "tx_2025": int(by_year.loc[2025, "transactions"]),
    "tx_multiple_2025_vs_2020": round(by_year.loc[2025, "transactions"] / by_year.loc[2020, "transactions"], 1),
    "offplan_share_2021_pct": round(by_year.loc[2021, "offplan_share_pct"], 1),
    "offplan_share_2025_pct": round(by_year.loc[2025, "offplan_share_pct"], 1),
    "median_psm_2020": int(round(by_year.loc[2020, "median_psm"])),
    "median_psm_2025": int(round(by_year.loc[2025, "median_psm"])),
    "median_psm_growth_2020_2025_pct": round((by_year.loc[2025, "median_psm"] / by_year.loc[2020, "median_psm"] - 1) * 100, 1),
    "median_size_2020_sqm": round(by_year.loc[2020, "median_size_sqm"], 1),
    "median_size_2025_sqm": round(by_year.loc[2025, "median_size_sqm"], 1),
    "median_size_2026ytd_sqm": round(by_year.loc[2026, "median_size_sqm"], 1),
    "median_size_2021_sqm": round(by_year.loc[2021, "median_size_sqm"], 1),
    "median_size_2026_is_lowest_since_2010": bool(by_year["median_size_sqm"].idxmin() == 2026),
    "offplan_psm_2025": int(round(reg25["Off-Plan"])),
    "ready_psm_2025": int(round(reg25["Ready"])),
    "offplan_premium_2025_pct": round((reg25["Off-Plan"] / reg25["Ready"] - 1) * 100, 1),
    "fastest_area_2021_2025": a.index[0],
    "fastest_area_growth_pct": round(a["growth_pct"].iloc[0], 1),
    "marina_growth_pct": round(a.loc["Marsa Dubai", "growth_pct"], 1),
    "n_areas_compared_2021_2025": int(len(a)),
    "busiest_area_2025": top_area_25.index[0],
    "busiest_area_2025_tx": int(top_area_25.iloc[0]),
    "busiest_area_2025_share_pct": round(top_area_25.iloc[0] / len(y25) * 100, 1),
    "share_2025_at_least_1m_aed_pct": round((y25["price_aed"] >= 1_000_000).mean() * 100, 1),
}
METRICS["facts"] = facts
METRICS["by_year"] = {int(k): {kk: (round(float(vv), 2)) for kk, vv in v.items()} for k, v in by_year.to_dict("index").items()}
METRICS["area_growth_2021_2025_top10"] = [
    {"area": i, "psm_2021": int(round(r.psm_2021)), "psm_2025": int(round(r.psm_2025)),
     "growth_pct": round(float(r.growth_pct), 1)}
    for i, r in a.head(10).iterrows()]
facts

In [ ]:
# Figure 1 - the boom in volume, driven by off-plan
fig, ax = plt.subplots()
p = df[df.year >= 2013].groupby(["year", "reg_type_label"]).size().unstack()[["Ready", "Off-Plan"]]
ax.bar(p.index, p["Ready"], color=style.ACCENT_2, label="Ready")
ax.bar(p.index, p["Off-Plan"], bottom=p["Ready"], color=style.ACCENT, label="Off-Plan")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
style.title(ax, f"Dubai home sales: {facts['tx_multiple_2025_vs_2020']}x more in 2025 than 2020 — mostly off-plan",
            "Registered residential sales per year (2026 = Jan to 2 Nov)")
ax.legend(loc="upper left"); ax.set_xticks(p.index)
style.source(fig); style.save(fig, R("figures", "01_volume_by_year.png"));
from IPython.display import Image; Image(R("figures", "01_volume_by_year.png"), width=760)

In [ ]:
# Figure 2 - price per sqm by year
fig, ax = plt.subplots()
q = df[df.year >= 2013].groupby(["year", "reg_type_label"])["psm"].median().unstack()
for k in ["Ready", "Off-Plan"]:
    ax.plot(q.index, q[k], marker="o", lw=2.5, color=style.SERIES[k], label=k)
allm = df[df.year >= 2013].groupby("year")["psm"].median()
ax.plot(allm.index, allm, ls="--", color=style.INK, lw=1.5, label="All")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
ax.set_ylabel("Median AED per sqm"); ax.set_xticks(q.index)
style.title(ax, f"Median price per sqm rose {facts['median_psm_growth_2020_2025_pct']}% from 2020 to 2025",
            "Median AED per square metre of registered residential sales (mix of homes changes every year)")
ax.legend(loc="upper left")
style.source(fig); style.save(fig, R("figures", "02_psm_by_year.png")); Image(R("figures", "02_psm_by_year.png"), width=760)

In [ ]:
# Figure 3 - which areas ran fastest
top = a.head(10).iloc[::-1]
g = top["growth_pct"]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top.index, g, color=style.ACCENT)
for y_, v in enumerate(g):
    ax.text(v + 2, y_, f"+{v:.0f}%", va="center", fontsize=10)
ax.set_xlabel("Change in median AED/sqm, 2021 → 2025")
ax.set_xlim(0, g.max() * 1.15)
style.title(ax, f"{facts['fastest_area_2021_2025']} led: +{facts['fastest_area_growth_pct']:.0f}% per sqm in four years",
            "Top 10 areas by price-per-sqm growth (areas with ≥300 sales in both 2021 and 2025; official DLD names)")
style.source(fig); style.save(fig, R("figures", "03_area_growth.png")); Image(R("figures", "03_area_growth.png"), width=760)

In [ ]:
# Figure 4 - homes are getting smaller
fig, ax = plt.subplots()
s = by_year.loc[2013:, "median_size_sqm"]
ax.plot(s.index, s.values, marker="o", lw=2.5, color=style.ACCENT_2)
for x_ in [2021, 2026]:
    ax.annotate(f"{s.loc[x_]:.0f} m²", (x_, s.loc[x_]), textcoords="offset points", xytext=(0, 10), ha="center")
ax.set_ylabel("Median size (m²)"); ax.set_xticks(s.index); ax.set_ylim(0, s.max() * 1.2)
assert facts["median_size_2026_is_lowest_since_2010"]
style.title(ax, f"Homes sold keep shrinking: {facts['median_size_2026ytd_sqm']:.0f} m² in 2026, the smallest since 2010",
            f"Median registered size of residential sales per year (2026 = Jan to 2 Nov)")
style.source(fig); style.save(fig, R("figures", "04_size_by_year.png")); Image(R("figures", "04_size_by_year.png"), width=760)

## 4. Features (leakage-safe) and the time split

History features (all computed from sales **strictly before** the registration day):
area / area×type×off-plan / area×rooms means of log(AED per sqm) over the past 365 or 90 days,
project means (365 days and all history), building means (all history), market means (30 days),
counts of those comparables, and project age. Plus the row's own facts: size, rooms, parking,
off-plan vs ready, Unit vs Villa, procedure group, area code, month.

In [ ]:
feat = build_features(df)
train, val, test = mdl.time_split(feat)
METRICS["splits"] = {
    "train": {"from": str(train.date.min().date()), "to": str(train.date.max().date()), "n": int(len(train))},
    "validation": {"from": str(val.date.min().date()), "to": str(val.date.max().date()), "n": int(len(val))},
    "test": {"from": str(test.date.min().date()), "to": str(test.date.max().date()), "n": int(len(test))},
    "n_features": len(FEATURES),
}
METRICS["splits"]

In [ ]:
# In-notebook leakage check (the full version lives in tests/test_no_leakage.py):
# recompute a few features by brute force using only earlier days and compare.
rng = np.random.RandomState(0)
sample = feat[feat.date >= "2018-01-01"].sample(40, random_state=0)
for _, row in sample.iterrows():
    past = feat[(feat.area_id == row.area_id) & (feat.date < row.date) &
                (feat.date >= row.date - pd.Timedelta(days=365))]
    exp = past[TARGET].mean() if len(past) else np.nan
    assert (np.isnan(exp) and np.isnan(row.area_mean_365)) or np.isclose(exp, row.area_mean_365), row.transaction_id
    mk = feat[(feat.date < row.date) & (feat.date >= row.date - pd.Timedelta(days=30))][TARGET].mean()
    assert np.isclose(mk, row.mkt_mean_30)
assert not {TARGET, "psm", "price_aed", "price_psf"} & set(FEATURES)
print("leakage check passed on", len(sample), "random rows")

## 5. Baselines and models (selection on the 2024 validation year only)

* **Market baseline** — last 30 days' average for off-plan/ready.
* **Comps rule (one strong feature)** — average of the same area + type + off-plan/ready over the past 365 days.
* **Building rule** — past sales in the same building, else same project, else the comps rule. A tough, hand-made benchmark.
* Ridge (linear), Random Forest (300k-row subsample), XGBoost, LightGBM.
  (PerpetualBooster was also tried, but its scores changed between two identical runs, so it was left out to keep every number reproducible.)

Target: log(AED per sqm). Main metric: **median absolute % error on the price (MdAPE)**, plus the share of
sales predicted within ±10% and ±20% (the usual way automated valuation models are judged).

In [ ]:
yv, yte, ytr = val[TARGET], test[TARGET], train[TARGET]
val_preds, test_preds, models, timings = {}, {}, {}, {}
for name, fn in [("Market baseline", mdl.baseline_market), ("Comps rule (1 feature)", mdl.baseline_comps),
                 ("Building rule", mdl.baseline_building)]:
    val_preds[name], test_preds[name] = fn(val), fn(test)

Xtr, Xva, Xte = mdl.X_tree(train), mdl.X_tree(val), mdl.X_tree(test)
Ctr, Cva, Cte = mdl.X_codes(train, train), mdl.X_codes(val, train), mdl.X_codes(test, train)

m, timings["Ridge (linear)"] = mdl.timed(mdl.make_linear().fit, train[FEATURES], ytr)
models["Ridge (linear)"] = m; val_preds["Ridge (linear)"] = m.predict(val[FEATURES]); test_preds["Ridge (linear)"] = m.predict(test[FEATURES])
m, timings["Random Forest"] = mdl.timed(mdl.fit_rf, Ctr, ytr)
models["Random Forest"] = m; val_preds["Random Forest"] = m.predict(Cva); test_preds["Random Forest"] = m.predict(Cte)
m, timings["XGBoost"] = mdl.timed(mdl.fit_xgb, Xtr, ytr, Xva, yv)
models["XGBoost"] = m; val_preds["XGBoost"] = m.predict(Xva); test_preds["XGBoost"] = m.predict(Xte)
m, timings["LightGBM"] = mdl.timed(mdl.fit_lgbm, Xtr, ytr, Xva, yv)
models["LightGBM"] = m; val_preds["LightGBM"] = m.predict(Xva); test_preds["LightGBM"] = m.predict(Xte)

val_table = pd.DataFrame({k: mdl.price_metrics(yv, p) for k, p in val_preds.items()}).T.sort_values("mdape_pct")
METRICS["validation"] = {k: {kk: round(vv, 3) if isinstance(vv, float) else vv for kk, vv in v.items()} for k, v in val_table.to_dict("index").items()}
display(val_table.round(2))
print("fit seconds:", timings)

## 6. Tune the best model family with Optuna (validation year only)

In [ ]:
ml_names = [k for k in val_table.index if k not in ("Market baseline", "Comps rule (1 feature)", "Building rule")]
best_family = val_table.loc[ml_names, "mdape_pct"].idxmin()
print("best family on validation:", best_family)
assert best_family in ("LightGBM", "XGBoost"), "tuning implemented for LightGBM (and XGBoost via the same pattern)"
N_TRIALS = 25
study = mdl.tune_lgbm(Xtr, ytr, Xva, yv, n_trials=N_TRIALS)
best = dict(study.best_params); best_iter = study.best_trial.user_attrs["best_iter"]
print(best, "| best iteration:", best_iter, "| val MdAPE:", round(study.best_value, 3))

tuned_tr = mdl.fit_lgbm(Xtr, ytr, Xva, yv, learning_rate=0.08, **best)
val_preds["LightGBM (tuned)"] = tuned_tr.predict(Xva)
METRICS["tuning"] = {"family": "LightGBM", "n_trials": N_TRIALS, "learning_rate": 0.08,
                     "best_params": {k: (round(v, 6) if isinstance(v, float) else v) for k, v in best.items()},
                     "best_iteration": int(best_iter),
                     "val_mdape_untuned": round(METRICS["validation"]["LightGBM"]["mdape_pct"], 3),
                     "val_mdape_tuned": round(float(study.best_value), 3)}
METRICS["validation"]["LightGBM (tuned)"] = {kk: round(vv, 3) if isinstance(vv, float) else vv
                                              for kk, vv in mdl.price_metrics(yv, val_preds["LightGBM (tuned)"]).items()}
METRICS["tuning"]

## 7. Final model → the untouched test period (2025-01-01 → 2026-11-02)

The tuned LightGBM is refit once on train + validation (≤ 2024) with the number of trees found on validation,
then scored on the test period. Every other model is also scored on test **for reference only** —
the choice was already made on validation.

In [ ]:
import lightgbm as lgb
trval = pd.concat([train, val])
final = lgb.LGBMRegressor(**mdl.lgbm_params(learning_rate=0.08, n_estimators=best_iter, **best))
final.fit(mdl.X_tree(trval), trval[TARGET])
test_preds["LightGBM (tuned, final)"] = final.predict(Xte)
test_table = pd.DataFrame({k: mdl.price_metrics(yte, p) for k, p in test_preds.items()}).T.sort_values("mdape_pct")
METRICS["test"] = {k: {kk: round(vv, 3) if isinstance(vv, float) else vv for kk, vv in v.items()} for k, v in test_table.to_dict("index").items()}
display(test_table.round(2))

fin, bld, cmp_ = (METRICS["test"]["LightGBM (tuned, final)"], METRICS["test"]["Building rule"],
                  METRICS["test"]["Comps rule (1 feature)"])
METRICS["headline"] = {
    "final_model": "LightGBM (tuned, final)",
    "test_mdape_pct": fin["mdape_pct"], "test_within_10pct": fin["within_10pct"], "test_within_20pct": fin["within_20pct"],
    "test_r2_log_psm": fin["r2_log_psm"],
    "building_rule_test_mdape_pct": bld["mdape_pct"], "comps_rule_test_mdape_pct": cmp_["mdape_pct"],
    "error_reduction_vs_building_rule_pct": round((1 - fin["mdape_pct"] / bld["mdape_pct"]) * 100, 1),
    "error_reduction_vs_comps_rule_pct": round((1 - fin["mdape_pct"] / cmp_["mdape_pct"]) * 100, 1),
    "beats_best_baseline_on_test": bool(fin["mdape_pct"] < min(bld["mdape_pct"], cmp_["mdape_pct"])),
    "val_mdape_tuned_pct": METRICS["tuning"]["val_mdape_tuned"],
}
METRICS["headline"]

### 7b. What if the model is retrained every month? (walk-forward)
The frozen model above never sees 2025-26 prices. In production you would retrain as new sales are registered.
Here, for each test month, the same tuned LightGBM is refit on **all sales before that month** and scores only that month.

In [ ]:
wf_pred = pd.Series(np.nan, index=test.index)
months = test.date.dt.to_period("M")
for per in sorted(months.unique()):
    start = per.start_time
    hist = feat[(feat.date >= "2013-01-01") & (feat.date < start)]
    mm = lgb.LGBMRegressor(**mdl.lgbm_params(learning_rate=0.08, n_estimators=best_iter, **best))
    mm.fit(mdl.X_tree(hist), hist[TARGET])
    idx = months[months == per].index
    wf_pred.loc[idx] = mm.predict(mdl.X_tree(test.loc[idx]))
test_preds["LightGBM retrained monthly (walk-forward)"] = wf_pred.to_numpy()
METRICS["walk_forward"] = {kk: round(vv, 3) if isinstance(vv, float) else vv
                           for kk, vv in mdl.price_metrics(yte, wf_pred.to_numpy()).items()}
METRICS["walk_forward"]

In [ ]:
# Does it hold up over time and across segments?
te = test.assign(pred=test_preds["LightGBM (tuned, final)"], bld=test_preds["Building rule"],
                 wf=test_preds["LightGBM retrained monthly (walk-forward)"])
te["ape_wf"] = np.abs(np.exp(te.wf) / te.psm - 1) * 100
te["ape"] = np.abs(np.exp(te.pred) / te.psm - 1) * 100
te["ape_bld"] = np.abs(np.exp(te.bld) / te.psm - 1) * 100
te["ym"] = te.date.dt.to_period("M").astype(str)
by_month = te.groupby("ym").agg(n=("ape", "size"), mdape_model=("ape", "median"), mdape_building_rule=("ape_bld", "median"),
                                mdape_walk_forward=("ape_wf", "median"))
te["has_bldg_history"] = np.where(te.bldg_n_all >= 3, "≥3 past sales in building", "new / unknown building")
seg = {}
for col in ["reg_type_label", "prop_type", "has_bldg_history"]:
    g = te.groupby(col, observed=True).agg(n=("ape", "size"), mdape_model=("ape", "median"), mdape_building_rule=("ape_bld", "median"))
    seg[col] = {str(k): {kk: round(float(vv), 2) for kk, vv in v.items()} for k, v in g.to_dict("index").items()}
METRICS["test_by_month"] = {k: {kk: round(float(vv), 2) for kk, vv in v.items()} for k, v in by_month.to_dict("index").items()}
METRICS["test_by_segment"] = seg
first3, last3 = by_month.mdape_model.iloc[:3].mean(), by_month.mdape_model.iloc[-3:].mean()
better = by_month.mdape_model < by_month.mdape_building_rule
worse_months = by_month.index[~better].tolist()
wf_better = by_month.mdape_walk_forward < by_month.mdape_building_rule
METRICS["drift"] = {"mdape_first3_months_avg": round(float(first3), 2), "mdape_last3_months_avg": round(float(last3), 2),
                    "first_month": by_month.index[0], "last_month": by_month.index[-1], "n_months": int(len(by_month)),
                    "frozen_months_better_than_building_rule": int(better.sum()),
                    "frozen_first_month_worse_than_rule": worse_months[0] if worse_months else None,
                    "walk_forward_months_better_than_building_rule": int(wf_better.sum()),
                    "walk_forward_last3_months_avg": round(float(by_month.mdape_walk_forward.iloc[-3:].mean()), 2),
                    "last_month_is_partial_days": int(test.date.max().day)}
display(by_month.round(2)); seg

## 8. How sure is it? Prediction intervals (regression → no probability calibration)

This is regression, so instead of a calibration curve we check whether stated uncertainty is honest:
an 80% interval built from the 2024 validation residuals (split conformal), then measured on 2025-26.

In [ ]:
half = mdl.split_conformal(yv - val_preds["LightGBM (tuned)"], alpha=0.2)
inside = np.abs(yte - test_preds["LightGBM (tuned, final)"]) <= half
by_m_cov = pd.Series(inside.values, index=te.ym).groupby(level=0).mean() * 100
METRICS["interval"] = {"nominal_coverage_pct": 80.0, "half_width_log": round(half, 4),
                       "relative_band_pct": [round((np.exp(-half) - 1) * 100, 1), round((np.exp(half) - 1) * 100, 1)],
                       "test_coverage_pct": round(float(inside.mean() * 100), 2),
                       "test_coverage_min_month_pct": round(float(by_m_cov.min()), 2)}
METRICS["interval"]

## 9. Explanation — SHAP

In [ ]:
import shap
shap_sample = test.sample(4000, random_state=mdl.SEED)
explainer = shap.TreeExplainer(final)
sv = explainer.shap_values(mdl.X_tree(shap_sample))
imp = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values(ascending=False)
METRICS["shap_mean_abs_top10"] = {k: round(float(v), 4) for k, v in imp.head(10).items()}
imp.head(10)

In [ ]:
FEATURE_LABELS = {
    "bldg_mean_all": "Past prices in same building", "proj_mean_365": "Same project, past 365d",
    "proj_mean_all": "Same project, all past", "seg_mean_365": "Area × type × off-plan, past 365d",
    "area_mean_365": "Area, past 365d", "area_mean_90": "Area, past 90d", "rooms_mean_365": "Area × rooms, past 365d",
    "log_area_sqm": "Size (log m²)", "rooms": "Bedrooms", "has_parking": "Parking included",
    "mkt_mean_30": "Whole market, past 30d", "mkt_reg_mean_30": "Market off-plan/ready, past 30d",
    "proj_age_days": "Project age (days)", "reg_type": "Off-plan vs ready", "prop_type": "Unit vs villa",
    "procedure_group": "DLD procedure", "area_id_cat": "Area code", "month": "Month",
    "area_n_365": "# area sales, past 365d", "seg_n_365": "# segment sales, past 365d",
    "proj_n_365": "# project sales, past 365d", "proj_n_all": "# project sales, all past", "bldg_n_all": "# building sales, all past",
}
fig, ax = plt.subplots(figsize=(10, 6))
t10 = imp.head(10).iloc[::-1]
ax.barh([FEATURE_LABELS.get(k, k) for k in t10.index], t10.values, color=[style.ACCENT if i >= 7 else style.MUTED for i in range(10)])
ax.set_xlabel("Mean |SHAP| (impact on log price per sqm)")
style.title(ax, "The model prices a home mostly by what its neighbours sold for",
            "Top 10 features by mean absolute SHAP value, 4,000 test-period sales")
style.source(fig); style.save(fig, R("figures", "08_shap_importance.png")); Image(R("figures", "08_shap_importance.png"), width=760)

## 10. Case studies — real test-period sales the model got right and wrong

In [ ]:
te["pred_price"] = np.exp(te.pred) * te.area_sqm
te["signed_err_pct"] = (te.pred_price / te.price_aed - 1) * 100
cand_hit = te[(te.bldg_n_all >= 3) & (te.ape < 1)].sort_values("transaction_id")
hit = cand_hit.iloc[len(cand_hit) // 2]
over = te[te.signed_err_pct > 0].sort_values("signed_err_pct")
over = over.iloc[int(len(over) * 0.99)]          # 99th percentile over-estimate, not the single worst row
under = te[te.signed_err_pct < 0].sort_values("signed_err_pct", ascending=False)
under = under.iloc[int(len(under) * 0.99)]       # 99th percentile under-estimate
cases = []
for label, r in [("hit", hit), ("over_estimate_p99", over), ("under_estimate_p99", under)]:
    s = explainer.shap_values(mdl.X_tree(te.loc[[r.name]]))[0]
    top = pd.Series(s, index=FEATURES).abs().sort_values(ascending=False).head(3).index
    xr = mdl.X_tree(te.loc[[r.name]]).iloc[0]
    cases.append({"case": label, "date": str(r.date.date()), "area": r.area_name_en,
                  "project": None if pd.isna(r.project_name) else r.project_name,
                  "type": r.property_type_label, "reg_type": r.reg_type_label,
                  "rooms": None if pd.isna(r.rooms_bucket) else int(r.rooms_bucket),
                  "size_sqm": round(float(r.area_sqm), 1), "actual_aed": int(r.price_aed),
                  "predicted_aed": int(round(r.pred_price, -3)), "error_pct": round(float(r.signed_err_pct), 1),
                  "past_sales_in_building": int(r.bldg_n_all), "procedure": r.procedure_name_en,
                  "top_shap_features": [FEATURE_LABELS.get(k, k) + (" (no data)" if pd.isna(xr[k]) else "") for k in top]})
METRICS["case_studies"] = cases
pd.DataFrame(cases).T

## 11. Result figures

In [ ]:
# Figure 5 - model comparison on validation + test
order = ["Market baseline", "Comps rule (1 feature)", "Ridge (linear)", "Building rule", "PerpetualBooster",
         "Random Forest", "XGBoost", "LightGBM"]
order = [o for o in order if o in METRICS["validation"]]
vals = [METRICS["validation"][o]["mdape_pct"] for o in order]
fig, ax = plt.subplots(figsize=(10, 6))
cols = [style.MUTED if ("rule" in o or "baseline" in o) else style.ACCENT_2 for o in order]
ax.barh(order[::-1], vals[::-1], color=cols[::-1])
for i, v in enumerate(vals[::-1]):
    ax.text(v + 0.3, i, f"{v:.1f}%", va="center", fontsize=10)
ax.set_xlabel("Median absolute % error on price (validation year 2024, lower is better)")
ax.set_xlim(0, max(vals) * 1.15)
style.title(ax, f"Boosted trees beat the toughest hand-made rule ({METRICS['validation']['Building rule']['mdape_pct']:.1f}% → {METRICS['validation']['LightGBM']['mdape_pct']:.1f}%)",
            "Grey = baselines/rules, teal = ML models (untuned). Selection was made on this year only")
style.source(fig); style.save(fig, R("figures", "05_model_comparison.png")); Image(R("figures", "05_model_comparison.png"), width=760)

In [ ]:
# Figure 6 - error over the test period: frozen vs retrained vs building rule
dr = METRICS["drift"]
fig, ax = plt.subplots()
x = np.arange(len(by_month))
ax.plot(x, by_month.mdape_building_rule, marker="o", color=style.MUTED, lw=2, label="Building rule (updates itself)")
ax.plot(x, by_month.mdape_model, marker="o", color=style.BAD, lw=2.2, label="Model frozen at end of 2024")
ax.plot(x, by_month.mdape_walk_forward, marker="o", color=style.ACCENT, lw=2.8, label="Model retrained every month")
ax.set_xticks(x[::2]); ax.set_xticklabels(by_month.index[::2], rotation=45, ha="right")
ax.set_ylabel("Median absolute % error"); ax.set_ylim(0, by_month[["mdape_model", "mdape_building_rule", "mdape_walk_forward"]].max().max() * 1.3)
ax.legend(loc="upper left")
style.title(ax, f"A frozen model ages: it loses to the simple rule from {dr['frozen_first_month_worse_than_rule']}",
            f"Median absolute % error per test month. Retrained monthly, the model beats the rule in {dr['walk_forward_months_better_than_building_rule']} of {dr['n_months']} months")
style.source(fig, f"Source: DLD open data via dubairealestatedata.com (CC BY 4.0). {dr['last_month']} has only {dr['last_month_is_partial_days']} days of data.")
style.save(fig, R("figures", "06_test_error_by_month.png")); Image(R("figures", "06_test_error_by_month.png"), width=760)

In [ ]:
# Figure 7 - predicted vs actual price per sqm (test)
fig, ax = plt.subplots(figsize=(8, 7))
hb = ax.hexbin(np.exp(yte) / 1000, np.exp(test_preds["LightGBM (tuned, final)"]) / 1000, gridsize=70, bins="log",
               cmap="YlOrBr", mincnt=1, extent=(0, 60, 0, 60))
ax.plot([0, 60], [0, 60], color=style.INK, lw=1, ls="--")
ax.set_xlim(0, 60); ax.set_ylim(0, 60)
ax.set_xlabel("Actual price per sqm (thousand AED)"); ax.set_ylabel("Predicted (thousand AED)")
fig.colorbar(hb, ax=ax, label="sales (log scale)")
style.title(ax, f"R² = {METRICS['headline']['test_r2_log_psm']:.2f} on log price per sqm",
            f"{METRICS['splits']['test']['n']:,} test-period sales; dashed line = perfect prediction")
style.source(fig); style.save(fig, R("figures", "07_pred_vs_actual.png")); Image(R("figures", "07_pred_vs_actual.png"), width=640)

In [ ]:
# Figure 9 - where it struggles
s = METRICS["test_by_segment"]
rows = [("Ready", s["reg_type_label"]["Ready"]), ("Off-Plan", s["reg_type_label"]["Off-Plan"]),
        ("Unit (apartment)", s["prop_type"]["Unit"]), ("Villa", s["prop_type"]["Villa"]),
        ("Building with ≥3 past sales", s["has_bldg_history"]["≥3 past sales in building"]),
        ("New / unknown building", s["has_bldg_history"]["new / unknown building"])]
fig, ax = plt.subplots(figsize=(10, 6))
labels = [r[0] for r in rows][::-1]; mv = [r[1]["mdape_model"] for r in rows][::-1]; bv = [r[1]["mdape_building_rule"] for r in rows][::-1]
yy = np.arange(len(rows))
ax.barh(yy + 0.2, bv, height=0.4, color=style.MUTED, label="Building rule")
ax.barh(yy - 0.2, mv, height=0.4, color=style.ACCENT, label="Model (frozen at end of 2024)")
for i in range(len(rows)):
    ax.text(mv[i] + 0.2, i - 0.2, f"{mv[i]:.1f}%", va="center", fontsize=9)
    ax.text(bv[i] + 0.2, i + 0.2, f"{bv[i]:.1f}%", va="center", fontsize=9, color="#5B6676")
ax.set_yticks(yy); ax.set_yticklabels(labels); ax.legend(loc="lower right")
ax.set_xlabel("Median absolute % error (test 2025-26)"); ax.set_xlim(0, max(bv + mv) * 1.2)
worst = max(rows, key=lambda r: r[1]["mdape_model"])
style.title(ax, f"{worst[0]} homes are the hardest to price ({worst[1]['mdape_model']:.1f}% error)",
            "Median absolute % error by segment, test 2025-26 (frozen final model vs building rule)")
style.source(fig); style.save(fig, R("figures", "09_error_by_segment.png")); Image(R("figures", "09_error_by_segment.png"), width=760)

In [ ]:
# Figure 10 - cleaning funnel
fig, ax = plt.subplots(figsize=(10, 5.6))
lg = pd.DataFrame(clean_log)
names = ["Raw file", "Quality flag = 0", "Residential unit/villa", "Home sub-types", "No exact duplicates", "From 2010"]
ax.barh(names[::-1], lg["rows_after"][::-1] / 1e6, color=[style.ACCENT] + [style.ACCENT_2] * (len(lg) - 1))
for i, v in enumerate(lg["rows_after"][::-1]):
    ax.text(v / 1e6 + 0.01, i, f"{v:,}", va="center", fontsize=10)
ax.set_xlabel("Rows (millions)"); ax.set_xlim(0, lg["rows_after"].max() / 1e6 * 1.2)
style.title(ax, f"From {METRICS['dataset']['rows_raw']:,} raw rows to {METRICS['dataset']['rows_clean']:,} clean home sales",
            "Every cleaning step and the rows left after it")
style.source(fig); style.save(fig, R("figures", "10_cleaning_funnel.png")); Image(R("figures", "10_cleaning_funnel.png"), width=760)

## 12. Save everything (metrics, model, data for the dashboard and the app)

In [ ]:
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, float) and np.isnan(o): return None
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o
METRICS["config"] = {"within_thresholds_pct": [10, 20], "min_area_sales_for_growth": 300, "rf_subsample_rows": 300000,
                     "optuna_trials": N_TRIALS, "interval_nominal_pct": 80, "shap_sample_rows": 4000,
                     "history_start": "2010-01-01", "model_start": "2013-01-01", "case_percentile": 99,
                     "min_past_sales_building_rule": 3, "seed": mdl.SEED}
with open(R("results", "metrics.json"), "w", encoding="utf-8") as fh:
    json.dump(_clean(METRICS), fh, indent=2, ensure_ascii=False, sort_keys=True)

final.booster_.save_model(R("models", "lgbm_final.txt"))   # frozen at end of 2024 (the evaluated model)
# production model for the app: same settings, refit on every sale up to the last day in the file
prod_rows = feat[feat.date >= "2013-01-01"]
prod = lgb.LGBMRegressor(**mdl.lgbm_params(learning_rate=0.08, n_estimators=best_iter, **best))
prod.fit(mdl.X_tree(prod_rows), prod_rows[TARGET])
prod.booster_.save_model(R("models", "lgbm_production.txt"))
cat_levels = {c: [str(x) for x in prod_rows[c].cat.categories] for c in CATEGORICAL}
json.dump({"features": FEATURES, "categorical": CATEGORICAL, "categories": cat_levels,
           "interval_half_width_log": METRICS["interval"]["half_width_log"],
           "production_trained_through": str(prod_rows.date.max().date())},
          open(R("models", "model_meta.json"), "w"), indent=1)

# test predictions (all rows, compact) for the dashboard / app explorer
out = te[["transaction_id", "date", "area_name_en", "project_name", "property_type_label", "reg_type_label",
          "rooms_bucket", "area_sqm", "price_aed", "pred_price", "signed_err_pct", "bldg_n_all"]].copy()
out["pred_price"] = out["pred_price"].round(0)
out["signed_err_pct"] = out["signed_err_pct"].round(2)
out.to_csv(R("results", "test_predictions.csv.gz"), index=False, compression="gzip")

# latest known history per area/segment for the what-if simulator (as of the last day in the data)
asof = feat["date"].max() + pd.Timedelta(days=1)
last = feat.copy(); last["all_key"] = 0
lk = (last[last.date > asof - pd.Timedelta(days=365)]
      .groupby(["area_id", "area_name_en", "property_type_label", "reg_type_label"], observed=True)[TARGET]
      .agg(seg_mean_365="mean", seg_n_365="size").reset_index())
area365 = last[last.date > asof - pd.Timedelta(days=365)].groupby("area_id")[TARGET].agg(area_mean_365="mean", area_n_365="size")
area90 = last[last.date > asof - pd.Timedelta(days=90)].groupby("area_id")[TARGET].agg(area_mean_90="mean")
rooms365 = (last[last.date > asof - pd.Timedelta(days=365)].groupby(["area_id", "rooms_key"])[TARGET].mean()
            .unstack().add_prefix("rooms_mean_365_"))
lk = lk.merge(area365, on="area_id", how="left").merge(area90, on="area_id", how="left").merge(rooms365, on="area_id", how="left")
mk = last[last.date > asof - pd.Timedelta(days=30)]
lk["mkt_mean_30"] = mk[TARGET].mean()
lk["mkt_reg_mean_30"] = lk["reg_type_label"].map(mk.groupby("reg_type_label")[TARGET].mean())
lk = lk[lk.seg_n_365 >= 20]
lk.to_csv(R("results", "whatif_lookup.csv"), index=False)
print("as of", asof.date(), "| what-if segments:", len(lk))
print(open(R("results", "metrics.json")).read()[:1500])

In [ ]:
# interactive offline dashboard
import make_dashboard
make_dashboard.build(ROOT)
print("dashboard written to", R("dashboard", "dubai_price_dashboard.html"))

## Limitations (read before trusting a number)
* The model predicts the **registered** price. It knows nothing about the view, floor, finishing, or a seller in a hurry.
* It leans on past sales nearby. A brand-new building with no history is clearly harder (see the segment chart).
* Medians in the EDA mix different homes each year (more small off-plan units recently), so a rise in the median
  is not the same as a like-for-like price index.
* Exact duplicate rows were dropped because identical bulk sales cannot be told apart from re-registrations.
* Data: Dubai Land Department open data via dubairealestatedata.com, CC BY 4.0. Code: MIT.